[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S08/S08_04_svm.ipynb)

# S08 · 04 — Máquinas de vectores de soporte (SVM)

**Tiempo estimado:** 55 min · **Sprint 8** · M3 (UD 3.8)

**Contexto TurisData.** Última familia del duelo: las **SVM**. Buscan la frontera que separa las clases con
el **mayor margen posible** y, con el «truco del núcleo» (*kernel*), pueden dibujar fronteras curvas.
Son potentes con pocos datos y muchas variables, pero **escalan mal** cuando hay muchas filas y son
poco explicables. Vamos a comprobarlo con las reservas.

**Al terminar sabrás:**
1. Explicar el margen, los vectores de soporte y qué hacen `C` y `gamma`.
2. Distinguir el kernel **lineal** del **RBF** (gaussiano).
3. Ajustar una SVM con validación cruzada y ver por qué **escalar es obligatorio**.
4. Valorar su coste de cómputo y su explicabilidad.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
SEMILLA = 42
sns.set_theme(style="whitegrid")

## 1. La idea en dos dimensiones

Imagina cada reserva como un punto en un plano (dos variables). Una SVM traza la **recta que separa
las dos clases dejando el mayor pasillo (margen) posible**. Los puntos que tocan el borde del pasillo
son los **vectores de soporte**: solo ellos definen la frontera; el resto de puntos podría
desaparecer sin que cambie.

- **`C`**: cuánto se penaliza un punto mal clasificado. `C` **grande** → margen estrecho, ajusta mucho
  (riesgo de sobreajuste). `C` **pequeño** → margen ancho y más tolerancia a errores.
- **Kernel**: si las clases no se separan con una recta, el kernel **RBF** «curva» el espacio.
  **`gamma`** controla cuánto influye cada punto: grande → fronteras muy locales y complejas.

Usamos un conjunto de juguete (dos «lunas») porque en 2D se ve la frontera.

In [ ]:
Xm, ym = make_moons(n_samples=400, noise=0.3, random_state=SEMILLA)
Xm_train, Xm_test, ym_train, ym_test = train_test_split(Xm, ym, test_size=0.3, random_state=SEMILLA)

def dibujar_frontera(modelo, ax, titulo):
    xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
    zz = modelo.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
    ax.scatter(Xm_train[:, 0], Xm_train[:, 1], c=ym_train, cmap="coolwarm", edgecolor="k", s=25)
    ax.set(title=titulo, xlabel="Variable 1", ylabel="Variable 2")

**Ejercicio 1.** Entrena en `Xm_train, ym_train` una SVM lineal (`svm_lineal = SVC(kernel="linear", C=1)`) y una
RBF (`svm_rbf = SVC(kernel="rbf", C=1, gamma=1)`). Guarda `acc_lineal` y `acc_rbf` (exactitud en
`Xm_test, ym_test` con `.score`) y en `n_soporte_rbf` el número de vectores de soporte de la RBF
(`svm_rbf.n_support_.sum()`).

In [ ]:
# TODO: SVC(kernel=...).fit(Xm_train, ym_train); .score(Xm_test, ym_test)
...
assert acc_rbf > acc_lineal, "En las dos lunas, el kernel RBF (curvo) debería superar al lineal"
assert 0 < n_soporte_rbf < len(Xm_train), "Los vectores de soporte son un subconjunto de los puntos"

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
dibujar_frontera(svm_lineal, ejes[0], f"Kernel lineal (exactitud test {acc_lineal:.2f})")
dibujar_frontera(svm_rbf, ejes[1], f"Kernel RBF (exactitud test {acc_rbf:.2f})")
plt.tight_layout()
plt.show()

### El efecto de `C` y `gamma`
Con RBF, mira cómo `gamma` alto **sobreajusta** (islotes alrededor de cada punto) y `gamma` bajo
**infraajusta** (frontera casi recta).

In [ ]:
fig, ejes = plt.subplots(1, 3, figsize=(15, 4))
for ax, g in zip(ejes, [0.05, 1, 50]):
    m = SVC(kernel="rbf", C=1, gamma=g).fit(Xm_train, ym_train)
    dibujar_frontera(m, ax, f"gamma = {g}  (train {m.score(Xm_train, ym_train):.2f} · test {m.score(Xm_test, ym_test):.2f})")
plt.tight_layout()
plt.show()

## 2. SVM con los datos de TurisData

Volvemos a las reservas y a `cancelada`. Las SVM calculan **distancias**: hay que **escalar** las
variables numéricas y codificar las categóricas. Para el AUC usamos `decision_function` (la distancia
a la frontera), que basta para ordenar las reservas por riesgo. (`probability=True` daría
probabilidades, pero entrena unas 5 veces más despacio.)

In [ ]:
df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["temporada_alta"] = df["fecha_llegada"].dt.month.isin([1, 2, 3, 7, 8, 12]).astype(int)
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X, y = df[columnas_num + columnas_cat], df["cancelada"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)


def prep_escalado():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), columnas_num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
    ])

def prep_sin_escalar():
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), columnas_num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), columnas_cat),
    ])

**Ejercicio 2.** Crea `svm_rbf_res = Pipeline([("prep", prep_escalado()), ("svm", SVC(kernel="rbf", C=1))])` y
`svm_rbf_sin = Pipeline([("prep", prep_sin_escalar()), ("svm", SVC(kernel="rbf", C=1))])`. Ajústalos con train
y guarda `auc_con_escalado` y `auc_sin_escalado`: el AUC en test con `decision_function`.

In [ ]:
# TODO: roc_auc_score(y_test, modelo.decision_function(X_test))
...
assert auc_con_escalado > auc_sin_escalado + 0.03, "Escalar debería mejorar claramente el AUC de la SVM"
assert auc_con_escalado > 0.70
print(f"AUC con escalado: {auc_con_escalado:.3f} · sin escalar: {auc_sin_escalado:.3f}")

## 3. Ajustar `C` y `gamma` con validación cruzada

`GridSearchCV` prueba todas las combinaciones de una rejilla. Como la SVM tarda, hacemos la búsqueda con
una **submuestra de 1500 reservas de train** (en un proyecto real, con más tiempo, usarías todas).

In [ ]:
sub_X = X_train.sample(1500, random_state=SEMILLA)
sub_y = y_train.loc[sub_X.index]
rejilla = {"svm__C": [0.1, 1, 10], "svm__gamma": [0.003, 0.01, 0.03]}
cv = StratifiedKFold(3, shuffle=True, random_state=SEMILLA)

**Ejercicio 3.** Crea `busqueda = GridSearchCV(Pipeline([("prep", prep_escalado()), ("svm", SVC(kernel="rbf"))]),
rejilla, cv=cv, scoring="roc_auc")` y ajústala con `sub_X, sub_y`. Guarda `mejores_params`
(`busqueda.best_params_`) y `mejor_auc_cv` (`busqueda.best_score_`).

In [ ]:
# TODO: GridSearchCV(pipeline, rejilla, cv=cv, scoring="roc_auc").fit(sub_X, sub_y)
...
assert set(mejores_params) == {"svm__C", "svm__gamma"}, "best_params_ debe contener C y gamma"
assert 0.65 < mejor_auc_cv < 0.85
print("Mejores parámetros:", mejores_params, f"· AUC CV = {mejor_auc_cv:.3f}")

In [ ]:
resultados = pd.DataFrame(busqueda.cv_results_)
tabla_grid = resultados.pivot_table(index="param_svm__C", columns="param_svm__gamma", values="mean_test_score").astype(float)
plt.figure(figsize=(5.5, 3.8))
sns.heatmap(tabla_grid, annot=True, fmt=".3f", cmap="Blues", cbar_kws={"label": "AUC (validación cruzada)"})
plt.title("SVM RBF: AUC según C y gamma")
plt.xlabel("gamma")
plt.ylabel("C")
plt.show()

## 4. El coste: las SVM no escalan con el número de filas

El tiempo de entrenamiento de una SVM crece **más que linealmente** (aproximadamente con el cuadrado del
número de filas). Con miles de reservas es asumible; con millones, no.

**Ejercicio 4.** Mide en `segundos` el tiempo de ajuste de `SVC(kernel="rbf", C=1, gamma=0.01)` sobre los
datos ya preparados `Z` con `tamanos = [500, 1000, 2000, 4000]` (las primeras `n` filas de `Z`). Guarda
en `factor_x4` cuántas veces más tarda con 4000 filas que con 1000 (`segundos[-1] / segundos[1]`).

In [ ]:
Z = prep_escalado().fit_transform(X_train)
yz = y_train.to_numpy()
tamanos = [500, 1000, 2000, 4000]
# TODO: para cada n: t0 = time.perf_counter(); SVC(...).fit(Z[:n], yz[:n]); guarda el tiempo
...
assert len(segundos) == 4, "Un tiempo por cada tamaño"
assert factor_x4 > 2.5, "Con 4 veces más filas debería tardar más de 2,5 veces más (crecimiento más que lineal)"
print("Segundos:", [round(s, 2) for s in segundos], f"· 4× filas → {factor_x4:.1f}× tiempo")

## 5. Mini-reto: tres SVM frente a frente

Crea `tabla_svm`, un DataFrame con filas `"lineal"`, `"rbf_defecto"` y `"rbf_ajustada"` y columnas
`auc_test`, `n_soporte` (nº de vectores de soporte) y `segundos_ajuste`. Usa `prep_escalado()`;
`rbf_defecto` es `SVC(kernel="rbf", C=1)` y `rbf_ajustada` usa los `mejores_params` de la búsqueda
(recuerda quitar el prefijo `svm__`). Entrena todo con el train completo.

In [ ]:
# TODO: tres pipelines; el nº de vectores de soporte está en pipeline["svm"].n_support_.sum()
...
assert list(tabla_svm.index) == ["lineal", "rbf_defecto", "rbf_ajustada"], "Filas en el orden pedido"
assert tabla_svm["auc_test"].between(0.65, 0.85).all()
assert (tabla_svm["n_soporte"] > 0).all()
tabla_svm

Fíjate en que la SVM **lineal** rinde igual o mejor que la RBF (los datos son casi «lineales» en el sentido
de la logística) y es más sencilla; y en que el **número de vectores de soporte** es una parte enorme del
entrenamiento: el modelo «guarda» muchas reservas, lo que encarece la predicción.

## 6. Para reflexionar

1. ¿Por qué solo los vectores de soporte definen la frontera?
2. ¿Qué pasa si subes `C` y `gamma` a la vez a valores muy grandes? ¿Cómo lo detectarías?
3. ¿Por qué una SVM no da probabilidades «de serie» y qué coste tiene pedirlas?
4. Si TurisData tuviera 5 millones de reservas, ¿usarías una SVM RBF? ¿Qué alternativa propondrías?

## 7. Resumen: qué has aprendido
- SVM = frontera de **margen máximo**; `C` controla la tolerancia a errores; el kernel RBF y `gamma` dan fronteras curvas.
- **Escalar es obligatorio**; `GridSearchCV` ajusta `C` y `gamma`.
- Coste alto con muchas filas, modelo poco explicable y sin probabilidades directas.
- Brillan con pocos datos, muchas variables y fronteras complejas.